# 1. `src/cv.py`: data, features and how a model is scored

This file does three jobs:

1. **Loads the data** for one task (`load_task`). Each task reads only its own folder under `data/official/dataset/`.
2. **Builds the feature table** a model sees (`basic_features`, `no_timing_features`, `task1_with_prefix_features`).
   `TASK_FEATURES` records which feature function each task uses in the submission.
3. **Scores a model** with repeated stratified 5-fold cross-validation on train + validation (`evaluate`):
   5 folds × 10 repeats = 50 scores per model, with the same folds for every model so they can be compared fairly.

The score is **macro F1**: F1 is computed per app and the six numbers are averaged, so a rare app like facebook
counts as much as a common one.

> Code cells marked **from `src/cv.py`** are copied from that file at commit `3b4c9c2`; the `.py` file is the one the project actually runs. Cells marked **demo** are extra examples for reading only.

In [1]:
# Setup: find the project root (the folder that holds data/official) and make src/ importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "official").exists())
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

project root: <project root>


## Constants

- `DATA`: where the challenge files are.
- `TASKS`: the two tasks. `task1_static` = statistics of the whole flow; `task2_realtime_prefix5` = statistics of the
  first 5 packets only.
- `DEFAULT_DROP`: columns never given to a model (`flow_id` and `split` are identifiers, `server_port` and
  `start_rel` are left out on purpose).
- `CV_SEED = 2026`: fixes which flows land in which fold.
- `FOLD_JOBS`: folds run in parallel processes (`-1` = all CPU cores).

In [2]:
# from src/cv.py
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import RepeatedStratifiedKFold

DATA = ROOT / "data" / "official" / "dataset"  # in src/cv.py: Path(__file__).resolve().parent.parent / ...
TASKS = ["task1_static", "task2_realtime_prefix5"]
DEFAULT_DROP = ["flow_id", "split", "server_port", "start_rel"]
CV_SEED = 2026  # fixes the fold assignment; identical across variants
# Folds run in parallel processes; models are single-threaded (models.MODEL_JOBS). On an M4 Pro one
# SoftVote fold takes 0.5 s single-threaded vs 3.3 s with n_jobs=-1 (experiment 007, step 0).
FOLD_JOBS = int(os.environ.get("MISSION_P_FOLD_JOBS", -1))

## Loading one task

`load_task(task, splits)` reads `<split>_features.csv` and `<split>_labels.csv` for each split and stacks them.
The `assert` checks that the features and labels are in the same flow order.

In [3]:
# from src/cv.py
def load_task(task, splits=("train", "validation")):
    """Return raw features (with flow_id) and labels for the given splits of ONE task."""
    Xs, ys = [], []
    for split in splits:
        X = pd.read_csv(DATA / task / f"{split}_features.csv")
        y = pd.read_csv(DATA / task / f"{split}_labels.csv")
        assert (X.flow_id.values == y.flow_id.values).all()
        Xs.append(X)
        ys.append(y["app"])
    return pd.concat(Xs, ignore_index=True), pd.concat(ys, ignore_index=True)

In [4]:
# demo: the 330 labelled flows of Task 2 (train + validation)
X_raw, y = load_task("task2_realtime_prefix5")
print(X_raw.shape)
print(y.value_counts().to_dict())
X_raw.head(3)

(330, 21)
{'google': 131, 'x': 73, 'youtube': 55, 'mozilla': 41, 'instagram': 19, 'facebook': 11}


,flow_id,split,start_rel,duration,transport,server_port,packet_count_up,packet_count_down,bytes_up,bytes_down,...,mean_pkt_size_down,std_pkt_size_up,std_pkt_size_down,mean_iat,std_iat,min_iat,max_iat,burst_count,up_down_packet_ratio,up_down_byte_ratio
0,flow_000003,train,0.280755,0.008733,udp,443,5,0,6470,0,...,0.0,0.000000,0.0,0.002183,0.003782,0.000000,0.008733,1,5.0,6470.000000
1,flow_000011,train,11.453972,0.057379,tcp,443,3,2,794,140,...,70.0,227.394420,4.0,0.014345,0.017193,0.001551,0.043424,1,1.5,5.671429
2,flow_000012,train,12.666567,0.101400,tcp,443,3,2,534,140,...,70.0,107.483332,4.0,0.025350,0.025256,0.001096,0.060069,1,1.5,3.814286


## Feature functions

- `basic_features`: drops `DEFAULT_DROP` and turns `transport` (tcp/udp) into 0/1. This gives **17 columns**.
- `no_timing_features`: also drops the 5 timing columns (`duration` and the inter-arrival times). This is what
  **Task 2** uses (12 columns): in the first 5 packets timing turned out to be noise (experiment 007).
- `task1_with_prefix_features`: what **Task 1** uses (34 columns). The 17 full-flow columns plus the 17 columns of
  the *same flow* from the Task 2 file, renamed with a `p5_` prefix (experiment 022). Reading Task 2 files for Task 1
  is allowed. The reverse, Task 1 data in Task 2, is forbidden by the challenge rules and never happens.
- `extra_data` on a function lists files outside the task's own folder, so `runlog` can record (hash) them.

In [5]:
# from src/cv.py
def basic_features(X, drop=DEFAULT_DROP):
    X = X.drop(columns=[c for c in drop if c in X.columns])
    X["transport"] = (X["transport"] == "udp").astype(int)
    return X


TIMING = ["duration", "mean_iat", "std_iat", "max_iat", "min_iat"]


def no_timing_features(X):
    """basic_features without the five timing columns (task 2 champion, experiment 007)."""
    return basic_features(X).drop(columns=TIMING)


_PREFIX_FILES = [f"task2_realtime_prefix5/{s}_features.csv" for s in ("train", "validation", "test")]
_prefix_cache = {}


def task1_with_prefix_features(X):
    """Task 1: basic_features + the Task 2 file's 17 basic columns for the same flow_id, prefixed p5_ (experiment 022).

    Task 2 files are used as input to Task 1 only. Condition 2 forbids the reverse (Task 1 features in Task 2),
    which never happens: Task 2 reads only its own folder. Disclosed in the technical report.
    """
    if "table" not in _prefix_cache:
        P = pd.concat([pd.read_csv(DATA / f) for f in _PREFIX_FILES], ignore_index=True)
        T = basic_features(P)
        T.columns = [f"p5_{c}" for c in T.columns]
        T.index = P.flow_id.values
        _prefix_cache["table"] = T
    B = basic_features(X).reset_index(drop=True)
    P = _prefix_cache["table"].loc[X.flow_id.values].reset_index(drop=True)
    return pd.concat([B, P], axis=1)


task1_with_prefix_features.extra_data = _PREFIX_FILES  # recorded (hashed) by runlog

# Feature set used by the submission for each task.
TASK_FEATURES = {"task1_static": task1_with_prefix_features, "task2_realtime_prefix5": no_timing_features}

In [6]:
# demo: the columns each task's model sees
X1_raw, _ = load_task("task1_static")
for task, X in (("task1_static", X1_raw), ("task2_realtime_prefix5", X_raw)):
    cols = list(TASK_FEATURES[task](X).columns)
    print(f"{task}: {TASK_FEATURES[task].__name__}, {len(cols)} columns")
    print("   ", cols, "\n")

task1_static: task1_with_prefix_features, 34 columns
    ['duration', 'transport', 'packet_count_up', 'packet_count_down', 'bytes_up', 'bytes_down', 'mean_pkt_size_up', 'mean_pkt_size_down', 'std_pkt_size_up', 'std_pkt_size_down', 'mean_iat', 'std_iat', 'min_iat', 'max_iat', 'burst_count', 'up_down_packet_ratio', 'up_down_byte_ratio', 'p5_duration', 'p5_transport', 'p5_packet_count_up', 'p5_packet_count_down', 'p5_bytes_up', 'p5_bytes_down', 'p5_mean_pkt_size_up', 'p5_mean_pkt_size_down', 'p5_std_pkt_size_up', 'p5_std_pkt_size_down', 'p5_mean_iat', 'p5_std_iat', 'p5_min_iat', 'p5_max_iat', 'p5_burst_count', 'p5_up_down_packet_ratio', 'p5_up_down_byte_ratio'] 

task2_realtime_prefix5: no_timing_features, 12 columns
    ['transport', 'packet_count_up', 'packet_count_down', 'bytes_up', 'bytes_down', 'mean_pkt_size_up', 'mean_pkt_size_down', 'std_pkt_size_up', 'std_pkt_size_down', 'burst_count', 'up_down_packet_ratio', 'up_down_byte_ratio'] 



## The evaluation loop

- `rf_factory(seed)`: a plain random forest, the default model for quick checks.
- `_run_fold`: trains on the training part of one fold and scores the held-out part. Optional `resample` changes only
  the training part.
- `evaluate`: makes the 50 folds, runs them in parallel and returns a dictionary with the per-fold scores, every
  held-out prediction (`y_pred`, `flow_id`, `fold`) and summaries. Fold `i` always uses model seed `i`, so two models
  see exactly the same folds and seeds. That is what makes a **paired** comparison between them valid (see notebook 4).
- `summarize`: one readable block per result.

In [7]:
# from src/cv.py
def rf_factory(seed):
    return RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=seed, n_jobs=1)


def _run_fold(i, tr, te, X, y, model_factory, resample, labels):
    t0 = time.perf_counter()
    X_tr, y_tr = X.iloc[tr], y.iloc[tr]
    if resample is not None:
        X_tr, y_tr = resample(X_tr, y_tr, i)
    pred = np.ravel(model_factory(i).fit(X_tr, y_tr).predict(X.iloc[te]))  # CatBoost returns (n, 1)
    return {"pred": pred, "seconds": time.perf_counter() - t0,
            "macro": f1_score(y.iloc[te], pred, average="macro"),
            "per_class": f1_score(y.iloc[te], pred, average=None, labels=labels, zero_division=0)}


def evaluate(task, features=basic_features, model_factory=rf_factory, resample=None,
             n_splits=5, n_repeats=10, cv_seed=CV_SEED, fold_jobs=FOLD_JOBS, splits=None, split_name=None):
    """Run repeated stratified K-fold CV on train+validation of `task`.

    `features(X_raw) -> DataFrame` (row-wise only), `model_factory(seed) -> estimator`,
    optional `resample(X_tr, y_tr, seed) -> (X_tr, y_tr)` applied to the training fold only.
    The model seed for fold i is i, so variants see identical folds and model seeds.
    Returns a dict with per-fold macro F1, per-fold per-class F1, out-of-fold predictions and summaries.
    """
    X_raw, y = load_task(task)
    X = features(X_raw)
    labels = sorted(y.unique())
    if splits is None:  # default: repeated stratified K-fold; `splits` may supply other folds (e.g. twin-aware, 036)
        splits = list(RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=cv_seed).split(X, y))
    out = Parallel(n_jobs=fold_jobs)(delayed(_run_fold)(i, tr, te, X, y, model_factory, resample, labels)
                                     for i, (tr, te) in enumerate(splits))
    macro = [o["macro"] for o in out]
    per_class = [o["per_class"] for o in out]
    seconds = [o["seconds"] for o in out]
    y_pred = [o["pred"] for o in out]
    y_true = [y.iloc[te].values for _, te in splits]
    flow_ids = [X_raw.flow_id.iloc[te].values for _, te in splits]
    folds = [np.full(len(te), i) for i, (_, te) in enumerate(splits)]
    macro, per_class = np.array(macro), np.array(per_class)
    repeat_means = macro.reshape(n_repeats, n_splits).mean(axis=1)
    return {
        "task": task,
        "n_flows": len(y),
        "features": list(X.columns),
        "labels": labels,
        "fold_macro": macro,
        "fold_per_class": per_class,
        "y_true": np.concatenate(y_true),  # pooled over all folds (each flow appears n_repeats times)
        "y_pred": np.concatenate(y_pred),
        "flow_id": np.concatenate(flow_ids),  # aligned with y_true / y_pred
        "fold": np.concatenate(folds),
        "fold_seconds": np.array(seconds),
        "extra_data": list(getattr(features, "extra_data", [])),  # files outside the task folder read by `features`  # fit + predict wall time per fold
        "cv": {"n_splits": n_splits, "n_repeats": n_repeats, "cv_seed": cv_seed,
               "resample": getattr(resample, "__name__", None) if resample else None,
               "fold_jobs": fold_jobs, "splits": split_name or "RepeatedStratifiedKFold"},
        "macro_mean": macro.mean(),
        "macro_std": macro.std(ddof=1),
        "repeat_std": repeat_means.std(ddof=1),
        "per_class_mean": dict(zip(labels, per_class.mean(axis=0))),
        "per_class_std": dict(zip(labels, per_class.std(axis=0, ddof=1))),
    }


def summarize(res):
    lines = [
        f"{res['task']}: macro F1 {res['macro_mean']:.4f} ± {res['macro_std']:.4f} (fold std), "
        f"repeat-mean std {res['repeat_std']:.4f}, {len(res['fold_macro'])} folds, {res['n_flows']} flows",
    ]
    for c in res["labels"]:
        lines.append(f"  {c:10s} {res['per_class_mean'][c]:.3f} ± {res['per_class_std'][c]:.3f}")
    return "\n".join(lines)

In [8]:
# demo: a quick run, random forest on Task 2 with 2 repeats (10 folds) instead of the default 10 repeats
res = evaluate("task2_realtime_prefix5", features=no_timing_features, model_factory=rf_factory, n_repeats=2)
print(summarize(res))

task2_realtime_prefix5: macro F1 0.7584 ± 0.0589 (fold std), repeat-mean std 0.0240, 10 folds, 330 flows
  facebook   0.520 ± 0.218
  google     0.869 ± 0.051
  instagram  0.670 ± 0.161
  mozilla    0.844 ± 0.106
  x          0.902 ± 0.071
  youtube    0.745 ± 0.082


When `src/cv.py` is run as a script it evaluates the random forest on both tasks with the default 50 folds:

```python
if __name__ == "__main__":
    for task in TASKS:
        print(summarize(evaluate(task)))
```

## Rest-of-flow features (experiment 030)

Task 1 describes the whole flow and Task 2 the first 5 packets of the same flow, so subtracting one from the other
gives exact statistics of **packets 6 to N**: counts, bytes, mean and spread of packet sizes per direction, and the
timing after packet 5. Sums subtract directly. Means and standard deviations are rebuilt from sums of squares.
`rest_of_flow` is used in experiments. The production model computes the same numbers from its own 34 columns
(`models.rest_from_columns`, notebook 2).

In [9]:
# from src/cv.py
def rest_of_flow(X):
    """Exact statistics of packets 6..N (after the 5-packet opening), from the Task 1 row and the Task 2 row of the
    same flow (experiment 030). Totals, means and population stds subtract exactly: count, bytes, mean and std of
    packet size per direction; duration after packet 5; mean and std of the remaining gaps. Valid for all 411 flows
    (no negative counts, bytes or variances). Directions/flows with no remaining packets get 0."""
    if "table" not in _prefix_cache:
        task1_with_prefix_features(X.head(1))
    if "rest" not in _prefix_cache:
        F = pd.concat([pd.read_csv(DATA / f"task1_static/{s}_features.csv") for s in ("train", "validation", "test")])
        F = F.set_index("flow_id")
        P = pd.concat([pd.read_csv(DATA / f) for f in _PREFIX_FILES]).set_index("flow_id").loc[F.index]
        out = {}
        for d in ("up", "down"):
            n = F[f"packet_count_{d}"] - P[f"packet_count_{d}"]
            b = F[f"bytes_{d}"] - P[f"bytes_{d}"]
            ss = (F[f"packet_count_{d}"] * (F[f"std_pkt_size_{d}"] ** 2 + F[f"mean_pkt_size_{d}"] ** 2)
                  - P[f"packet_count_{d}"] * (P[f"std_pkt_size_{d}"] ** 2 + P[f"mean_pkt_size_{d}"] ** 2))
            m = np.where(n > 0, b / np.where(n > 0, n, 1), 0.0)
            v = np.where(n > 0, ss / np.where(n > 0, n, 1) - m ** 2, 0.0)
            out.update({f"r_count_{d}": n, f"r_bytes_{d}": b, f"r_mean_size_{d}": m, f"r_std_size_{d}": np.sqrt(np.clip(v, 0, None))})
        nf = F.packet_count_up + F.packet_count_down
        g = nf - 5
        dur = F.duration - P.duration
        ssg = (nf - 1) * (F.std_iat ** 2 + F.mean_iat ** 2) - 4 * (P.std_iat ** 2 + P.mean_iat ** 2)
        mg = np.where(g > 0, dur / np.where(g > 0, g, 1), 0.0)
        vg = np.where(g > 0, ssg / np.where(g > 0, g, 1) - mg ** 2, 0.0)
        out.update({"r_duration": dur, "r_mean_iat": mg, "r_std_iat": np.sqrt(np.clip(vg, 0, None))})
        _prefix_cache["rest"] = pd.DataFrame(out, index=F.index)
    return _prefix_cache["rest"].loc[X.flow_id.values].reset_index(drop=True)


def task1_prefix_rest_features(X):
    """022 Task 1 features (17 + 17 prefix-5) + 11 rest-of-flow columns (experiment 030, rest_add)."""
    return pd.concat([task1_with_prefix_features(X), rest_of_flow(X)], axis=1)


def task1_rest_features(X):
    """17 Task 1 columns + 11 rest-of-flow columns, no raw prefix-5 columns (experiment 030, rest_swap)."""
    return pd.concat([basic_features(X).reset_index(drop=True), rest_of_flow(X)], axis=1)


task1_prefix_rest_features.extra_data = _PREFIX_FILES
task1_rest_features.extra_data = _PREFIX_FILES

In [10]:
# demo: the 11 rest-of-flow columns for three flows
rest_of_flow(X1_raw.head(3))

,r_count_up,r_bytes_up,r_mean_size_up,r_std_size_up,r_count_down,r_bytes_down,r_mean_size_down,r_std_size_down,r_duration,r_mean_iat,r_std_iat
0,128,47782,373.296875,501.385181,130,12055,92.730769,13.037690,242.063573,0.938231,3.120335
1,6,590,98.333333,54.862454,4,3145,786.250000,518.654208,0.184879,0.018488,0.021572
2,51,5575,109.313725,190.124984,35,7840,224.000000,337.075320,277.214085,3.223420,7.027714
